# AI-Based Crop Recommendation System
### Run every cell from top to bottom (Shift+Enter)
---

In [ ]:
# CELL 1 - Install libraries
import subprocess
subprocess.run(['pip', 'install', 'pandas', 'numpy', 'scikit-learn', 'matplotlib', 'seaborn', 'joblib', '-q'])
print('All libraries ready!')

In [ ]:
# CELL 2 - Import all libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import joblib
import os
import warnings
warnings.filterwarnings('ignore')
print('All imports successful!')

In [ ]:
# CELL 3 - Generate dataset (2200 rows, 22 crops, no internet needed)
np.random.seed(42)

crops = {
    'rice':        dict(N=(60,100), P=(30,60),  K=(30,60),  temp=(20,27), hum=(80,90), ph=(5.5,7.0), rain=(150,300)),
    'maize':       dict(N=(60,100), P=(50,80),  K=(40,70),  temp=(18,27), hum=(55,75), ph=(5.5,7.5), rain=(50,100)),
    'chickpea':    dict(N=(10,40),  P=(50,90),  K=(60,100), temp=(15,25), hum=(14,30), ph=(5.5,7.5), rain=(30,60)),
    'kidneybeans': dict(N=(10,30),  P=(50,80),  K=(15,30),  temp=(15,25), hum=(18,22), ph=(5.5,7.0), rain=(20,30)),
    'pigeonpeas':  dict(N=(10,30),  P=(50,80),  K=(15,30),  temp=(25,35), hum=(40,70), ph=(5.0,7.0), rain=(60,100)),
    'mothbeans':   dict(N=(10,30),  P=(30,60),  K=(15,30),  temp=(25,38), hum=(40,60), ph=(3.5,6.5), rain=(30,60)),
    'mungbean':    dict(N=(10,40),  P=(30,60),  K=(15,30),  temp=(25,35), hum=(80,90), ph=(6.0,7.5), rain=(30,60)),
    'blackgram':   dict(N=(10,40),  P=(50,80),  K=(15,30),  temp=(25,35), hum=(60,80), ph=(6.0,7.5), rain=(30,60)),
    'lentil':      dict(N=(10,30),  P=(50,80),  K=(15,30),  temp=(15,25), hum=(60,70), ph=(5.5,7.5), rain=(30,60)),
    'pomegranate': dict(N=(10,30),  P=(10,20),  K=(30,50),  temp=(18,27), hum=(80,95), ph=(5.5,7.0), rain=(100,200)),
    'banana':      dict(N=(80,120), P=(50,80),  K=(40,80),  temp=(25,35), hum=(75,90), ph=(5.5,7.0), rain=(100,200)),
    'mango':       dict(N=(10,30),  P=(10,30),  K=(25,45),  temp=(24,35), hum=(50,70), ph=(5.0,7.0), rain=(60,100)),
    'grapes':      dict(N=(10,30),  P=(100,140),K=(190,230),temp=(8,23),  hum=(80,90), ph=(5.5,7.5), rain=(60,100)),
    'watermelon':  dict(N=(80,120), P=(10,20),  K=(40,60),  temp=(24,35), hum=(80,90), ph=(5.5,7.0), rain=(40,60)),
    'muskmelon':   dict(N=(80,120), P=(10,20),  K=(40,60),  temp=(28,38), hum=(90,95), ph=(6.0,7.5), rain=(20,30)),
    'apple':       dict(N=(0,20),   P=(100,140),K=(190,230),temp=(0,15),  hum=(90,95), ph=(5.5,7.5), rain=(100,200)),
    'orange':      dict(N=(0,20),   P=(5,15),   K=(5,15),   temp=(10,20), hum=(90,95), ph=(6.0,8.0), rain=(100,200)),
    'papaya':      dict(N=(40,60),  P=(10,20),  K=(40,60),  temp=(25,35), hum=(90,95), ph=(6.5,8.0), rain=(100,200)),
    'coconut':     dict(N=(0,20),   P=(0,10),   K=(25,45),  temp=(25,35), hum=(80,90), ph=(5.0,7.0), rain=(100,200)),
    'cotton':      dict(N=(100,140),P=(15,30),  K=(15,30),  temp=(24,35), hum=(75,85), ph=(6.0,8.0), rain=(60,100)),
    'jute':        dict(N=(60,100), P=(35,60),  K=(35,60),  temp=(24,37), hum=(70,90), ph=(6.0,7.5), rain=(150,250)),
    'coffee':      dict(N=(80,120), P=(15,30),  K=(25,45),  temp=(15,27), hum=(55,75), ph=(6.0,7.0), rain=(150,250)),
}

rows = []
for crop, r in crops.items():
    for _ in range(100):
        rows.append({
            'N':           np.random.randint(*r['N']),
            'P':           np.random.randint(*r['P']),
            'K':           np.random.randint(*r['K']),
            'temperature': round(np.random.uniform(*r['temp']), 2),
            'humidity':    round(np.random.uniform(*r['hum']),  2),
            'ph':          round(np.random.uniform(*r['ph']),   2),
            'rainfall':    round(np.random.uniform(*r['rain']), 2),
            'label':       crop
        })

df = pd.DataFrame(rows).sample(frac=1, random_state=42).reset_index(drop=True)
print(f'Dataset shape    : {df.shape}')
print(f'Unique crops     : {df["label"].nunique()}')
print(f'Columns          : {df.columns.tolist()}')
df.head()

In [ ]:
# CELL 4 - Explore the data
print('=== Basic Statistics ===')
print(df.describe().round(2))
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Samples per Crop ===')
print(df['label'].value_counts())

In [ ]:
# CELL 5 - Bar chart: samples per crop
plt.figure(figsize=(14, 5))
colors = plt.cm.Set3(np.linspace(0, 1, 22))
df['label'].value_counts().plot(kind='bar', color=colors, edgecolor='white')
plt.title('Number of samples per crop', fontsize=14)
plt.xlabel('Crop name')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# CELL 6 - Heatmap: feature correlation
plt.figure(figsize=(9, 6))
numeric_cols = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
sns.heatmap(df[numeric_cols].corr(), annot=True, fmt='.2f',
            cmap='coolwarm', center=0, square=True, linewidths=0.5)
plt.title('Correlation between soil and weather features', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# CELL 7 - Histograms: distribution of each feature
feature_colors = {'N':'#4CAF50','P':'#2196F3','K':'#FF9800',
                  'temperature':'#F44336','humidity':'#9C27B0',
                  'ph':'#795548','rainfall':'#00BCD4'}
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()
for i, (col, color) in enumerate(feature_colors.items()):
    axes[i].hist(df[col], bins=30, color=color, alpha=0.7, edgecolor='white')
    axes[i].set_title(col, fontsize=11)
    axes[i].axvline(df[col].mean(), color='black', linestyle='--', linewidth=1.2)
axes[7].axis('off')
plt.suptitle('Distribution of all 7 input features', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# CELL 8 - Prepare features X and labels y
X = df[['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']]
y = df['label']

# Encode crop names to numbers (ML needs numbers, not strings)
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print('Crop name to number mapping:')
for i, crop in enumerate(le.classes_):
    print(f'  {i:2d} = {crop}')
print(f'\nX shape: {X.shape}  |  y shape: {y_encoded.shape}')

In [ ]:
# CELL 9 - Split data: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)
print(f'Training samples : {X_train.shape[0]} rows (80%)')
print(f'Testing samples  : {X_test.shape[0]} rows (20%)')
print('Model will only see training data. Test data stays hidden until evaluation.')

In [ ]:
# CELL 10 - Train Random Forest model
print('Training model... (takes about 10 seconds)')
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)
print('Training COMPLETE!')
print(f'Trees built : {model.n_estimators}')
print(f'Crops learned : {model.n_classes_}')

In [ ]:
# CELL 11 - Evaluate accuracy
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f'Model Accuracy : {accuracy * 100:.2f}%')
print(f'Correct        : {int(accuracy * len(y_test))} / {len(y_test)}')
print()
print('=== Per-crop accuracy ===')
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
# CELL 12 - Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(16, 13))
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrRd',
            xticklabels=le.classes_, yticklabels=le.classes_,
            linewidths=0.5)
plt.title('Confusion Matrix', fontsize=14)
plt.xlabel('Predicted crop')
plt.ylabel('Actual crop')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()
print('Diagonal = correct predictions. Off-diagonal = mistakes.')

In [ ]:
# CELL 13 - Feature importance
feature_names = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
feat_series = pd.Series(model.feature_importances_, index=feature_names).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
feat_series.plot(kind='barh', ax=ax, color='#2196F3', edgecolor='white')
for i, val in enumerate(feat_series.values):
    ax.text(val + 0.002, i, f'{val:.3f}', va='center', fontsize=10)
ax.set_title('Which input feature matters most for crop prediction?', fontsize=13)
ax.set_xlabel('Importance score')
plt.tight_layout()
plt.show()

print('Ranking:')
for rank, (name, score) in enumerate(feat_series.sort_values(ascending=False).items(), 1):
    print(f'  {rank}. {name:<14} {score:.4f}  {"#" * int(score*100)}')

In [ ]:
# CELL 14 - Prediction function
def predict_crop(N, P, K, temperature, humidity, ph, rainfall, location=''):
    input_df = pd.DataFrame([[N, P, K, temperature, humidity, ph, rainfall]],
                             columns=['N','P','K','temperature','humidity','ph','rainfall'])
    pred      = model.predict(input_df)[0]
    proba     = model.predict_proba(input_df)[0]
    top3_idx  = np.argsort(proba)[::-1][:3]
    top3      = [(le.classes_[i], round(proba[i]*100, 1)) for i in top3_idx]
    crop_name = le.inverse_transform([pred])[0]
    confidence = round(max(proba)*100, 1)

    print('=' * 52)
    if location: print(f'  Location  : {location}')
    print(f'  Input     : N={N} P={P} K={K} Temp={temperature}C')
    print(f'              Humidity={humidity}% pH={ph} Rain={rainfall}mm')
    print('=' * 52)
    print(f'  BEST CROP : {crop_name.upper()}')
    print(f'  Confidence: {confidence}%')
    print()
    print('  Top 3 options:')
    for medal, (crop, pct) in zip(['1st','2nd','3rd'], top3):
        print(f'    {medal}  {crop:<14} {pct}%  {"#" * int(pct/5)}')
    print('=' * 52)
    return crop_name

# Test with 3 Indian locations
print('FARMER 1 - Nashik, Maharashtra')
predict_crop(90, 42, 43, 24.0, 82.0, 6.5, 202.9, 'Nashik, Maharashtra')

print('\nFARMER 2 - Ludhiana, Punjab')
predict_crop(120, 55, 50, 19.0, 65.0, 7.0, 68.0, 'Ludhiana, Punjab')

print('\nFARMER 3 - Wayanad, Kerala')
predict_crop(100, 18, 30, 26.5, 87.0, 6.2, 180.0, 'Wayanad, Kerala')

In [ ]:
# CELL 15 - Try YOUR OWN values here!
# Change these values to match your location and run the cell

MY_N           = 80      # Nitrogen (kg/ha)
MY_P           = 40      # Phosphorus (kg/ha)
MY_K           = 40      # Potassium (kg/ha)
MY_TEMPERATURE = 25.5    # Temperature in Celsius
MY_HUMIDITY    = 80.0    # Humidity in %
MY_PH          = 6.5     # Soil pH
MY_RAINFALL    = 150.0   # Rainfall in mm
MY_LOCATION    = 'Pune, Maharashtra'

predict_crop(MY_N, MY_P, MY_K, MY_TEMPERATURE,
             MY_HUMIDITY, MY_PH, MY_RAINFALL, MY_LOCATION)

In [ ]:
# CELL 16 - Save the trained model to disk
os.makedirs('model', exist_ok=True)
joblib.dump(model, 'model/crop_model.pkl')
joblib.dump(le,    'model/label_encoder.pkl')
print('Model saved    : model/crop_model.pkl')
print('Encoder saved  : model/label_encoder.pkl')
print(f'Model size     : {os.path.getsize("model/crop_model.pkl")/1024:.1f} KB')

In [ ]:
# CELL 17 - Reload model from disk and verify it works
loaded_model = joblib.load('model/crop_model.pkl')
loaded_le    = joblib.load('model/label_encoder.pkl')

test_df = pd.DataFrame([[90, 42, 43, 24.0, 82.0, 6.5, 202.9]],
                        columns=['N','P','K','temperature','humidity','ph','rainfall'])
pred = loaded_model.predict(test_df)[0]
print(f'Loaded model prediction : {loaded_le.inverse_transform([pred])[0]}')
print('Model reloaded and working correctly!')

In [ ]:
# CELL 18 - Write Flask API code to app.py
flask_code = '''from flask import Flask, request, jsonify
from flask_cors import CORS
import joblib, numpy as np, pandas as pd

app = Flask(__name__)
CORS(app)

model = joblib.load("model/crop_model.pkl")
le    = joblib.load("model/label_encoder.pkl")

@app.route("/", methods=["GET"])
def home():
    return jsonify({"status": "running"})

@app.route("/predict", methods=["POST"])
def predict():
    try:
        d = request.json
        df = pd.DataFrame([[d["N"],d["P"],d["K"],d["temperature"],
                            d["humidity"],d["ph"],d["rainfall"]]],
                           columns=["N","P","K","temperature","humidity","ph","rainfall"])
        pred  = model.predict(df)[0]
        proba = model.predict_proba(df)[0]
        top3  = [{"crop": le.classes_[i], "confidence": round(float(proba[i])*100,1)}
                  for i in np.argsort(proba)[::-1][:3]]
        return jsonify({
            "recommended_crop": le.inverse_transform([pred])[0],
            "confidence": round(float(max(proba))*100,1),
            "top_3": top3
        })
    except Exception as e:
        return jsonify({"error": str(e)}), 400

if __name__ == "__main__":
    app.run(debug=True, port=5000)
'''
with open('app.py', 'w') as f:
    f.write(flask_code)
print('app.py saved!')
print('To run: pip install flask flask-cors  then  python app.py')

In [ ]:
# CELL 19 - Project summary
print('=' * 55)
print('  AI CROP RECOMMENDATION - PROJECT SUMMARY')
print('=' * 55)
print(f'  Accuracy     : {accuracy_score(y_test, model.predict(X_test))*100:.2f}%')
print(f'  Total crops  : 22')
print(f'  Dataset rows : 2200')
print(f'  Trees built  : 100')
print()
print('  Files saved:')
for f in ['model/crop_model.pkl','model/label_encoder.pkl','app.py']:
    if os.path.exists(f):
        print(f'    {f}')
print()
print('  Next steps:')
print('  1 -> Run app.py as Flask server')
print('  2 -> Build React or Flutter frontend')
print('  3 -> Connect Bhuparikshan API (soil data)')
print('  4 -> Connect Agrinex API (market prices)')
print('  5 -> Add Sawan TTS (voice in Marathi/Hindi)')
print('  6 -> Deploy on Railway.app (free hosting)')
print('=' * 55)